# RF-Diffusion Quick Start

This notebook demonstrates the RF-Diffusion evaluation API using mock data.
It shows how to load the evaluation pipeline and compute SSIM and SNR metrics
without needing the full upstream model weights.

## 1. Imports and Setup

In [ ]:
import sys
import numpy as np
import torch

sys.path.insert(0, "/path/to/RF-Diffusion")

from src.evaluation import compute_ssim, compute_snr, aggregate_metrics

## 2. Generate Mock Data

We create synthetic complex-valued signals that mimic the shape of real
Wi-Fi CSI data (512 time-domain samples with real and imaginary channels).

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

n_samples = 5
signal_length = 512
device = torch.device("cpu")

# Ground-truth signal: random complex values
truth_real = torch.randn(n_samples, signal_length, dtype=torch.float32)
truth_imag = torch.randn(n_samples, signal_length, dtype=torch.float32)

# Simulated prediction: ground truth + small noise
pred_real = truth_real + 0.05 * torch.randn_like(truth_real)
pred_imag = truth_imag + 0.05 * torch.randn_like(truth_imag)

print(f"Generated {n_samples} mock signal pairs, shape {signal_length}")

## 3. Compute SSIM

SSIM (Structural Similarity Index) measures structural similarity between
the generated and ground-truth signals. Values range from 0 to 1; higher is better.

The `compute_ssim` function expects complex-valued tensors in the format
(channels, height, width) — matching the time-frequency input dimension.

In [ ]:
ssim_values = []
input_dim = signal_length  # treated as both height and width in the implementation

for i in range(n_samples):
    pred_complex = torch.complex(pred_real[i], pred_imag[i])  # (512,)
    truth_complex = torch.complex(truth_real[i], truth_imag[i])  # (512,)
    # Reshape to (channels, height, width) = (1, dim, dim) as expected
    pred_tensor = torch.stack([pred_real[i], pred_imag[i]]).unsqueeze(1)  # (2, 1, 512)
    truth_tensor = torch.stack([truth_real[i], truth_imag[i]]).unsqueeze(1)  # (2, 1, 512)
    # SSIM internally reshapes to (dim, dim), so input_dim should match
    ssim = compute_ssim(pred_tensor, truth_tensor, sample_rate=20, input_dim=input_dim, device=device)
    ssim_values.append(ssim)
    print(f"  Sample {i}: SSIM = {ssim:.4f}")

print(f"\nMean SSIM: {np.mean(ssim_values):.4f}")

## 4. Compute SNR (for MIMO / channel estimation)

SNR (Signal-to-Noise Ratio) measures the power ratio between signal and noise.
For 5G MIMO channel estimation, higher SNR indicates better channel reconstruction.

The `compute_snr` function accepts NumPy arrays with the last dimension
containing [real, imag] pairs.

In [ ]:
snr_values = []
for i in range(n_samples):
    pred_np = np.stack([pred_real[i].numpy(), pred_imag[i].numpy()], axis=-1)
    truth_np = np.stack([truth_real[i].numpy(), truth_imag[i].numpy()], axis=-1)
    snr_db = compute_snr(pred_np, truth_np)
    snr_values.append(snr_db)
    print(f"  Sample {i}: SNR = {snr_db:.2f} dB")

print(f"\nMean SNR: {np.mean(snr_values):.2f} dB")

## 5. Aggregate Metrics

Use `aggregate_metrics` to summarize a list of per-sample values into
mean, std, min, max, and count.

In [ ]:
summary = aggregate_metrics(ssim_values, snr_values)
print("Aggregated results:")
for metric_name, stats in summary.items():
    print(f"\n  {metric_name}:")
    for stat_name, value in stats.items():
        print(f"    {stat_name}: {value:.4f}")

## 6. Understanding the Outputs

### SSIM (Structural Similarity Index)
- **Range**: 0 to 1 (1 = perfect structural match)
- **Interpretation**: Captures structural similarity between signals, more aligned
  with human perception than pixel-wise MSE. SSIM ≈ 0.81 matches the paper's reported value.

### SNR (Signal-to-Noise Ratio)
- **Range**: -∞ to +∞ dB (higher is better)
- **Interpretation**: Ratio of signal power to noise power. For 5G MIMO, SNR ≈ 29.95 dB
  matches the reproduced result.

### Output Files
When running full inference, the pipeline saves `.mat` files to `results/raw/<task>/samples/`:
- `pred`: Generated signal
- `data`: Ground-truth signal

Metrics are written to `results/metrics/` as JSON files.

## 7. Running Full Experiments

To run the full evaluation pipeline with real model weights, first set up the upstream:

```bash
bash scripts/setup_upstream.sh
pip install -r requirements.txt
```

Then run:

```bash
# Wi-Fi evaluation
python scripts/run_wifi_sampling.py --num-samples 5 --strategy native

# 5G MIMO evaluation
python scripts/run_5g_mimo.py --num-samples 5

# Generate figures
python scripts/plot_results.py
```